# Bitcoin Plain Talk — Swahili QLoRA fine-tune

Track B MVP: fine-tune a small open model on the Bitcoin Plain Talk Swahili
glossary dataset, on a free Colab T4 GPU, $0 cost.

**Fully open-source stack** (no non-commercial or restricted licenses):

| Component | License | Notes |
|---|---|---|
| Base model: Qwen3-1.7B | Apache 2.0 | Swapped in for InkubaLM — InkubaLM is CC BY-NC 4.0, which disqualifies it once we need a genuinely open-source stack. |
| Unsloth (fine-tuning) | Apache 2.0 | Core `pip install unsloth` package only (not the AGPL-3.0 Unsloth Studio GUI). |
| transformers / trl / peft | Apache 2.0 | Standard Hugging Face training stack. |
| Dataset | CC BY-SA 4.0 | `datasets/bitcoin-plain-talk-sw.jsonl` in this repo. |

Google Colab and the Hugging Face Hub are used purely as free *compute and
hosting* — neither is open-source software itself, but that doesn't affect
the license of the model, adapter, or dataset we produce and publish.

**Before running:** this dataset is currently tiny (88 examples / 22 terms).
That's enough to prove the fine-tuning pipeline works end-to-end and to spot
check whether the model picks up Bitcoin-Swahili phrasing it didn't have
before — it is **not** enough data to expect deep generalization. Treat the
eval at the bottom as a directional check, not a benchmark.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps --force-reinstall "git+https://github.com/unslothai/unsloth.git"

In [ ]:
from unsloth import FastLanguageModel
import torch

# Swap to "unsloth/Qwen3-4B-unsloth-bnb-4bit" for a bigger/slower/better model,
# or "unsloth/Qwen3-0.6B-unsloth-bnb-4bit" for a faster/smaller one.
MODEL_NAME = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,       # auto-detect on T4
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Load the dataset and hold out a small eval set

We hold out every example for four terms entirely, so the eval afterwards is
on genuinely unseen questions rather than examples the model trained on.

In [ ]:
import json
import urllib.request

# Once datasets/ is published to Hugging Face, replace this cell with:
#   from datasets import load_dataset
#   raw = load_dataset("<your-org>/bitcoin-plain-talk-sw", split="train")
#   examples = list(raw)
#
# Until then, pull the JSONL straight from the GitHub repo (push the
# datasets/ folder to main first) or upload the file to this Colab session
# manually and point DATA_PATH at it.
RAW_URL = "https://raw.githubusercontent.com/wandiamugo/bitcoin-plain-talk/main/datasets/bitcoin-plain-talk-sw.jsonl"

try:
    with urllib.request.urlopen(RAW_URL) as f:
        lines = f.read().decode("utf-8").strip().split("\n")
except Exception as e:
    raise RuntimeError(
        "Could not fetch the dataset from GitHub. Either push datasets/ to "
        "main first, or upload bitcoin-plain-talk-sw.jsonl to this Colab "
        "session and set DATA_PATH to its local path instead."
    ) from e

examples = [json.loads(l) for l in lines if l.strip()]

HOLDOUT_TERMS = {"UTXO", "Mempool", "Seed Phrase", "Lightning Network"}

train_examples = [e for e in examples if e["term_en"] not in HOLDOUT_TERMS]
eval_examples = [e for e in examples if e["term_en"] in HOLDOUT_TERMS]

print(f"{len(train_examples)} training examples, {len(eval_examples)} held out for eval")

In [ ]:
from datasets import Dataset

def to_chatml(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_examples).map(to_chatml)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,          # small dataset — watch eval outputs for repetition/overfitting
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer.train()

## Save and push

Pushing requires `huggingface-cli login` (or `HF_TOKEN` set) in this session
— a manual, one-time step tied to your own HF account/namespace.

In [ ]:
HF_REPO = "<your-org>/bitcoin-plain-talk-qwen3-1.7b-sw-lora"

model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# Uncomment once you're ready to publish:
# model.push_to_hub(HF_REPO, token="hf_...")
# tokenizer.push_to_hub(HF_REPO, token="hf_...")

## Evaluate: base model vs. fine-tuned, on held-out questions

This is the comparison that actually backs the "we trained a model" claim —
run the *same held-out instructions* through the base model and the
fine-tuned one and read the outputs side by side.

In [ ]:
FastLanguageModel.for_inference(model)

def generate(instruction, max_new_tokens=200):
    messages = [{"role": "user", "content": instruction}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

for ex in eval_examples[:8]:
    print("INSTRUCTION:", ex["instruction"])
    print("EXPECTED (source glossary):", ex["output"][:200])
    print("FINE-TUNED MODEL:", generate(ex["instruction"]))
    print("-" * 80)